# 使用 Microsoft Foundry 微調模型

本筆記本遵循目前的 [Microsoft Foundry 微調工作流程](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning?WT.mc_id=academic-105485-koreyst)。它使用指向您的 Foundry 資源 `/openai/v1/` 端點的 **OpenAI Python SDK (v1)**，因此相同程式碼在更改客戶端設置後也可用於 OpenAI 平台。

> <strong>前置條件</strong>
> - 一個具有 **Foundry Owner** 角色（需要微調和部署）的 [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) 專案。
> - `pip install "openai>=1.0" python-dotenv`
> - 一個包含 `AZURE_OPENAI_ENDPOINT` 和 `AZURE_OPENAI_API_KEY` 的 `.env` 文件（請參閱 [課程設置指南](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)）。
> - 目前支援的基礎模型，例如 `gpt-4.1-mini`（請參閱 [可微調模型列表](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models)）。

微調透過在與您的任務相關的額外範例上重新訓練基礎模型來提升其能力。提示工程技術如 _few-shot learning_ 和 _retrieval-augmented generation_ 會以相關資料強化提示，但受限於模型的上下文窗口。

透過微調，您重新訓練模型本身（使用的範例數量遠超過上下文窗口大小）並部署一個 _自訂_ 版本，在推理時不再需要那些範例。這可提升品質、釋放上下文窗口，並透過縮短提示降低成本和延遲。技術底層，Foundry 使用 **LoRA（低秩適應）** 以進行高效訓練。

Foundry 支援三種技術：本教程使用的 **監督式微調（SFT）**，另有 **DPO**（偏好對齊）和 **RFT**（強化微調）。流程包含四步：

1. 準備並上傳您的訓練 <strong>和驗證</strong> 資料。
2. 執行訓練工作以建立微調模型。
3. 監控工作，審查指標，並選擇檢查點。
4. 部署微調模型並用於推理。

本教程將使用 SFT 對 `gpt-4.1-mini` 進行微調，建立一個以打油詩回答週期表相關問題的聊天機器人。

---


### 第 1.1 步：準備你的數據集

我們來構建一個聊天機械人，透過用打油詩回答有關元素的問題，幫助你理解元素週期表。在_這個_簡單的教學中，我們只會創建一個數據集，用幾個回應的範例展示數據預期的格式，以訓練模型。在實際應用中，你需要創建一個包含更多範例的數據集。你也可能能使用（若存在）針對你應用領域的開放數據集，並將其重新格式化以用於微調。

由於我們聚焦於 `gpt-4.1-mini`，並尋求單輪回應（對話完成），我們可以使用[這個建議的格式](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst)來創建範例，該格式符合 OpenAI 聊天完成的要求。如果你預期多輪對話內容，則會使用[多輪範例格式](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst)，其中包含一個 `weight` 參數，用來指示哪些訊息應該在微調過程中被使用（或不使用）。

在我們這裡的教學中，我們將使用較簡單的單輪格式。數據為[jsonl 格式](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst)，每行一條記錄，每條記錄用 JSON 格式的物件表示。下面的片段顯示了兩條記錄作為範例—完整的範例集（10 個示例）可見於 [training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl) ，我們將用於微調教學。<strong>注意：</strong>每條記錄_必須_定義在單行中（而非像格式化 JSON 文件那樣分拆多行）

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

在實際應用中，你需要更大規模的範例集以取得良好效果——這會在回應品質與微調所需的時間/成本間做出取捨。我們使用較小的集合，是為了快速完成微調以示範此過程。想了解更複雜的微調教學，請參閱[此 OpenAI Cookbook 範例](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst)。


---

### 步驟 1.2：上傳您的數據集

使用 Files API (`purpose="fine-tune"`) 將您的訓練和驗證檔案上傳到 Foundry。提供 <strong>驗證檔案</strong> 可讓 Foundry 報告驗證損失，幫助您發現過擬合。

在執行下方程式碼之前，請確保您已經：
 - 安裝 SDK：`pip install "openai>=1.0" python-dotenv`
 - 建立 `.env` 檔案並設定 `AZURE_OPENAI_ENDPOINT` 和 `AZURE_OPENAI_API_KEY`

該程式碼會建立一個指向您 Foundry 資源 `/openai/v1/` 端點的 OpenAI 用戶端，然後上傳兩個檔案並印出它們的 ID。


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Point the OpenAI SDK at your Microsoft Foundry resource's /openai/v1/ endpoint.
# (For the OpenAI platform instead, use: client = OpenAI()  with OPENAI_API_KEY set.)
client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
)

training_response = client.files.create(
    file=open("./training-data.jsonl", "rb"), purpose="fine-tune"
)
validation_response = client.files.create(
    file=open("./validation-data.jsonl", "rb"), purpose="fine-tune"
)

training_file_id = training_response.id
validation_file_id = validation_response.id

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)


---

### 步驟 2.1：使用 SDK 建立微調工作


In [ ]:
# Create the fine-tuning job.
# trainingType "GlobalStandard" is the recommended tier (lower cost, faster queue).
# Use "Standard" if you need regional data residency, or "Developer" for cheap experiments.
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini",              # base model to fine-tune
    training_file=training_file_id,
    validation_file=validation_file_id,
    suffix="elements-limerick",        # helps you identify the resulting model
    seed=105,                          # makes the run reproducible
    extra_body={"trainingType": "GlobalStandard"},
)

job_id = job.id
print("Fine-tuning Job ID:", job_id)
print("Status:", job.status)


---

### 步驟 2.2：檢查工作狀態

以下是您可以使用 `client.fine_tuning.jobs` API 執行的幾件事：
- `client.fine_tuning.jobs.list(limit=<n>)` - 列出最近的 n 個微調工作
- `client.fine_tuning.jobs.retrieve(<job_id>)` - 獲取特定工作的詳細資訊
- `client.fine_tuning.jobs.cancel(<job_id>)` - 取消工作
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<n>)` - 列出該工作中的事件
- `client.fine_tuning.jobs.checkpoints.list(<job_id>)` - 列出可部署的檢查點（最後幾個 epoch）

工作開始時，Foundry 會先_驗證訓練檔案_，以確保資料格式正確。隨後開始訓練，時間長短取決於模型和數據集大小，可能從幾分鐘到幾小時不等。


In [ ]:
# List the last 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the current state of your fine-tuning job
client.fine_tuning.jobs.retrieve(job_id)

# List up to 10 events from the job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)


In [ ]:
# Track the job status until it is complete
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)


---

### 步驟 2.3：追蹤事件以監察進度


In [ ]:
# Track progress in a more granular way by checking events.
# Re-run this cell until you see "The job has successfully completed".
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)


In [ ]:
# When the job finishes, the last few epochs are available as deployable checkpoints.
# Best practice: don't blindly deploy the final epoch - review the checkpoints and pick
# the one that generalizes best (watch train_loss vs. valid_loss and token accuracy).
checkpoints = client.fine_tuning.jobs.checkpoints.list(job_id)
for cp in checkpoints.data:
    print(cp.fine_tuned_model_checkpoint, "| step:", cp.step_number)


### 步驟 2.4：在 Foundry 入口網站檢閱狀態、指標和檢查點


你亦可以在 [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) 內的 **Build > Fine-tuning** 中追蹤作業。選擇你的作業以查看其狀態、訓練事件、超參數和指標。留意以下指標：

- `train_loss` 與 `full_valid_loss` — 應該隨時間下降。
- `train_mean_token_accuracy` 與 `full_valid_mean_token_accuracy` — 應該上升。

如果訓練和驗證曲線出現分歧，可能是過度擬合 — 試試減少訓練輪數或使用較小學習率乘數。下方插圖展示了你會見到的狀態、訊息和指標面板類型（具體用戶介面因提供者而異）。

![Fine-tuning job status](../../../../../translated_images/zh-HK/fine-tuned-model-status.563271727bf7bfba.webp)


您亦可以向下滾動視覺化儀表板進一步查看狀態訊息與指標，如圖所示：

| 訊息 | 指標 |
|:---|:---|
| ![Messages](../../../../../translated_images/zh-HK/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/zh-HK/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### 步驟 3.1：取得微調模型編號

當工作成功後，`response.fine_tuned_model` 會保存自定義模型的編號（例如，`gpt-4.1-mini-2025-04-14.ft-...`）。在 Azure 上，您必須先<strong>部署</strong>該模型，才能調用它。


In [ ]:
# Retrieve the id of the fine-tuned model once the job has succeeded
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)


### 步驟 3.2：部署微調模型

與 OpenAI 平台（可直接調用微調模型 ID）不同，Microsoft Foundry 需要您先<strong>部署</strong>模型。最簡單的方法是使用 [Foundry 入口網站](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)：前往 **建置 > 微調**，選擇已完成的工作，然後選擇 <strong>部署</strong>。為部署命名（例如 `elements-limerick`）— 此部署名稱即為您從程式碼中調用的名稱。

您也可以使用控制平面 REST/ARM API 以程式方式部署；請參閱[部署指南](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning-deploy?WT.mc_id=academic-105485-koreyst)。請記住，部署後的自訂模型將按小時計費，且非活動的部署會在 15 天後被移除。


In [ ]:
# Once deployed, call your fine-tuned model by its DEPLOYMENT name via the Responses API.
# (On the OpenAI platform you'd pass fine_tuned_model_id directly instead.)
deployment_name = "elements-limerick"  # the name you gave the deployment in Foundry

completion = client.responses.create(
    model=deployment_name,
    input=[
        {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
        {"role": "user", "content": "Tell me about Strontium"},
    ],
    store=False,
)
print(completion.output_text)


---

### 第 3.3 步：在 Foundry playground 測試你的微調模型

你也可以在 [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) **Playground** 中測試已部署的模型 - 從模型下拉選單中選擇你的微調部署並嘗試輸入提示。請使用你訓練時使用的 <strong>相同系統訊息</strong>；不同的系統訊息可能會改變模型行為。

> **提示：** 將微調模型與基礎的 `gpt-4.1-mini` 並排比較。注意微調模型會依據你的範例以打油詩格式回應，而基礎模型則只是遵循系統提示。

**這是一個刻意簡化的範例來展示流程，而非真實世界的資料集。** 在生產環境中，你會基於真實資料進行微調（例如，用於客服的產品目錄），此時品質差異會更加明顯——且僅透過提示工程要達到同等品質，每次呼叫的代幣成本會高得多。想要更完整的範例，請參考 [OpenAI Cookbook 的微調指南](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) 以及 [Foundry 微調教學](https://learn.microsoft.com/azure/ai-foundry/openai/tutorials/fine-tune?WT.mc_id=academic-105485-koreyst)。

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
